# CNN Regression — Multi-Dataset Comparison

Trains the **same CNN** on **7 preprocessing variants** of the same 213-image dataset
and compares performance using the project's primary metrics.

| # | Dataset | Description |
|---|---------|-------------|
| 0 | **nocc** | Raw 16-bit images, no colour filtering |
| 1 | **K=0.5** | CIELAB statistical filter  [μ − 0.5σ, μ + 0.5σ] |
| 2 | **K=1.0** | CIELAB statistical filter  [μ − σ,   μ + σ]   |
| 3 | **K=1.5** | CIELAB statistical filter  [μ − 1.5σ, μ + 1.5σ] |
| 4 | **K=2.0** | CIELAB statistical filter  [μ − 2σ,   μ + 2σ]   |
| 5 | **K=3.0** | CIELAB statistical filter  [μ − 3σ,   μ + 3σ]   |
| 6 | **Z-score** | LAB Z-score filter  \|z\| ≤ 2.5 per channel |

**Primary evaluation metrics:** Accuracy@0 (exact match), Accuracy@1 (within-1), R², RMSE

**Key guarantees**
- Train/val split computed **once** — reused for all 7 variants (zero leakage).
- Augmentation applied **only** to the training set.
- 128×128 with **crop** (Resize→148 then RandomCrop/CenterCrop), not naive resize.
- `skimage.color` imported as `skcolor` to avoid loop variable collisions.

## 1. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from glob import glob
import os
from PIL import Image
from tqdm import tqdm
from skimage import io as skio
from skimage import color as skcolor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
mpl.rcParams['figure.figsize'] = 16, 8
plt.rcParams['font.size'] = 10

torch.manual_seed(12)
np.random.seed(12)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Libraries imported successfully!')

## 2. Configure Parameters

In [ ]:
# ── Path ────────────────────────────────────────────────────────────────────
input_folder = (
    r'F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT\nocc\CSVS01'
)

# ── Dataset variants ─────────────────────────────────────────────────────────
# (label, K)  K=None → raw nocc;  K='zscore' → Z-score filter;  K=float → stat filter
DATASETS = [
    ('nocc',    None),
    ('K=0.5',   0.5),
    ('K=1.0',   1.0),
    ('K=1.5',   1.5),
    ('K=2.0',   2.0),
    ('K=3.0',   3.0),
    ('zscore', 'zscore'),
]

# ── Folder naming ────────────────────────────────────────────────────────────
folder_numbers  = list(range(1, 9))   # 1 → 8
FOLDER_NAME_TPL = 'Score={n:02d}'
use_cc          = False

# ── Image / training hyper-parameters ───────────────────────────────────────
CROP_SIZE                = 128        # final spatial size fed to the CNN
RESIZE_BEFORE_CROP       = 148        # resize short-side to this before cropping
batch_size               = 32
num_epochs               = 1000
learning_rate            = 0.001
train_split              = 0.8
early_stopping_patience  = 30
ZSCORE_THRESHOLD         = 2.5        # |z| cut-off for the z-score filter

# kept for backward-compat references inside the code
img_size = (CROP_SIZE, CROP_SIZE)

# ── Colours for 7 datasets ───────────────────────────────────────────────────
DS_COLORS = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860','#DA8BC3']

print('Parameters:')
print(f'  input_folder      : {input_folder}')
print(f'  datasets          : {[d for d, _ in DATASETS]}')
print(f'  crop size         : {CROP_SIZE}  (resize to {RESIZE_BEFORE_CROP} first)')
print(f'  batch_size        : {batch_size}')
print(f'  num_epochs        : {num_epochs}')
print(f'  learning_rate     : {learning_rate}')
print(f'  train_split       : {train_split}')
print(f'  early_stopping    : {early_stopping_patience} epochs')
print(f'  zscore_threshold  : {ZSCORE_THRESHOLD}')

if os.path.exists(input_folder):
    print(f'\nFolders found under input_folder:')
    for d in sorted(os.listdir(input_folder)):
        print(f'  {d}')
else:
    print(f'\n⚠  input_folder does not exist: {input_folder}')

## 3. Helper Functions

### 3.1 Load Image Paths

In [ ]:
def find_and_filter_images(folder, folder_numbers, use_cc=False):
    """
    Scan Score=NN sub-folders and return [(path, float_target), ...].
    Robustly handles missing folders and datasets without cc/nocc tokens.
    """
    cc_token   = '+cc+'   if use_cc else '+nocc+'
    anti_token = '+nocc+' if use_cc else '+cc+'
    data_list  = []

    for fn in folder_numbers:
        dir_name = FOLDER_NAME_TPL.format(n=fn)
        fp = os.path.join(folder, dir_name)
        if not os.path.exists(fp):
            print(f'  ⚠ folder not found: {fp}')
            continue

        all_tifs = [p for p in glob(os.path.join(fp, '*.tif'))
                    if os.path.isfile(p)]
        has_tokens = any(cc_token in p or anti_token in p for p in all_tifs)
        paths = [p for p in all_tifs if cc_token in p] if has_tokens else all_tifs

        for p in paths:
            data_list.append((p, float(fn)))
        print(f'  {dir_name}: {len(paths)} images  (target = {fn})')

    return data_list


print('Scanning dataset...')
full_data_list = find_and_filter_images(input_folder, folder_numbers, use_cc)
print(f'\nTotal images found: {len(full_data_list)}')

targets_all = [t for _, t in full_data_list]
unique_cls, counts_cls = np.unique(targets_all, return_counts=True)
plt.figure(figsize=(10, 4))
plt.bar(unique_cls, counts_cls, color='steelblue', edgecolor='black', linewidth=1.5)
plt.xlabel('Score (folder)', fontsize=12, fontweight='bold')
plt.ylabel('Images', fontsize=12, fontweight='bold')
plt.title('Target distribution', fontsize=13, fontweight='bold')
plt.xticks(unique_cls)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### 3.2 Master Train / Val Split *(computed once — no data leakage)*

Split is fixed here from the raw path list and reused for all 7 variants.

In [ ]:
assert len(full_data_list) > 0, 'No images found — check input_folder and FOLDER_NAME_TPL'

train_list_master, val_list_master = train_test_split(
    full_data_list,
    test_size=1 - train_split,
    stratify=[t for _, t in full_data_list],
    random_state=42
)

_train_paths = {p for p, _ in train_list_master}
_val_paths   = {p for p, _ in val_list_master}
assert _train_paths.isdisjoint(_val_paths), 'DATA LEAKAGE: train and val share paths!'

print(f'Total  : {len(full_data_list)}')
print(f'Train  : {len(train_list_master)}')
print(f'Val    : {len(val_list_master)}')
print(f'Overlap: 0  ✓')

print('\nClass distribution:')
print(f'{"Score":>7}  {"Train":>6}  {"Val":>5}')
for cls in unique_cls:
    n_tr = sum(1 for _, t in train_list_master if t == cls)
    n_vl = sum(1 for _, t in val_list_master   if t == cls)
    print(f'{int(cls):>7}  {n_tr:>6}  {n_vl:>5}')

### 3.3a CIELAB Statistical Pixel Filter (K-sigma band)

In [ ]:
def lab_statistical_filter(img_uint16, K):
    """
    Zero out pixels whose L*, a*, or b* value falls outside
    [mean - K*std, mean + K*std] computed over foreground pixels.

    Parameters
    ----------
    img_uint16 : np.ndarray  (H, W, 3)  dtype uint16
    K          : float

    Returns
    -------
    filtered   : np.ndarray  (H, W, 3)  float32 in [0, 1]
    """
    img_f   = img_uint16.astype(np.float32) / 65535.0
    fg_mask = img_f.sum(axis=2) > 0
    lab     = skcolor.rgb2lab(img_f)

    keep_mask = np.ones(img_f.shape[:2], dtype=bool)
    for c in range(3):
        ch      = lab[:, :, c]
        fg_vals = ch[fg_mask]
        if fg_vals.size == 0:
            continue
        mu, sigma = fg_vals.mean(), fg_vals.std()
        keep_mask &= (ch >= mu - K * sigma) & (ch <= mu + K * sigma)

    result = img_f.copy()
    result[~(fg_mask & keep_mask)] = 0.0
    return result


print('lab_statistical_filter defined.')

### 3.3b Z-Score Pixel Filter

In [ ]:
def zscore_filter(img_uint16, threshold=2.5):
    """
    Remove pixels where the absolute Z-score in ANY LAB channel exceeds
    `threshold`.  Unlike the K-sigma band (which clips a symmetric interval
    around the mean), this rejects outlier pixels based on their individual
    standardised distance — equivalent to K-sigma but applied per-pixel
    across all three channels jointly (logical OR rejection).

    Parameters
    ----------
    img_uint16 : np.ndarray  (H, W, 3)  dtype uint16
    threshold  : float  default 2.5

    Returns
    -------
    filtered   : np.ndarray  (H, W, 3)  float32 in [0, 1]
    """
    img_f   = img_uint16.astype(np.float32) / 65535.0
    fg_mask = img_f.sum(axis=2) > 0
    lab     = skcolor.rgb2lab(img_f)

    keep_mask = np.ones(img_f.shape[:2], dtype=bool)
    for c in range(3):
        ch      = lab[:, :, c]
        fg_vals = ch[fg_mask]
        if fg_vals.size == 0:
            continue
        mu, sigma = fg_vals.mean(), fg_vals.std()
        if sigma < 1e-6:
            continue
        z = np.abs((ch - mu) / sigma)
        keep_mask &= (z <= threshold)

    result = img_f.copy()
    result[~(fg_mask & keep_mask)] = 0.0
    return result


print(f'zscore_filter defined  (threshold = {ZSCORE_THRESHOLD}).')

### 3.4 Dataset Class

In [ ]:
class ImageRegressionDataset(Dataset):
    """
    Loads 16-bit TIFF images and applies the selected pixel filter:
      K=None     → raw (nocc)
      K='zscore' → zscore_filter
      K=float    → lab_statistical_filter(K)
    """
    def __init__(self, data_list, transform=None, target_noise_std=0.0, K=None):
        self.data_list        = data_list
        self.transform        = transform
        self.target_noise_std = target_noise_std
        self.K                = K

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, target = self.data_list[idx]
        try:
            img_array = skio.imread(img_path)
            if img_array.dtype != np.uint16:
                raise ValueError(
                    f'{os.path.basename(img_path)} is not 16-bit (dtype={img_array.dtype})')

            if self.K == 'zscore':
                img_f = zscore_filter(img_array, threshold=ZSCORE_THRESHOLD)
            elif self.K is not None:
                img_f = lab_statistical_filter(img_array, self.K)
            else:
                img_f = img_array.astype(np.float32) / 65535.0

            image = torch.from_numpy(img_f.transpose(2, 0, 1)).float()

            if self.transform:
                image = self.transform(image)

            target_t = torch.tensor([target], dtype=torch.float32)
            if self.target_noise_std > 0:
                target_t = target_t + torch.randn_like(target_t) * self.target_noise_std
            return image, target_t

        except Exception as e:
            print(f'  Error loading {img_path}: {e}')
            image = torch.zeros(3, img_size[0], img_size[1])
            return image, torch.tensor([target], dtype=torch.float32)


print('ImageRegressionDataset defined.')

### 3.5 Transforms — Resize → Crop (128×128)

Training: `Resize(148)` → `RandomCrop(128)` → flips/rotations → Normalize  
Validation: `Resize(148)` → `CenterCrop(128)` → Normalize

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(RESIZE_BEFORE_CROP),        # short-side → 148
    transforms.RandomCrop(CROP_SIZE),              # random 128×128 patch
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomChoice([
        transforms.Lambda(lambda x: torch.rot90(x, k=0, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=1, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=2, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=3, dims=[1, 2])),
    ]),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize(RESIZE_BEFORE_CROP),         # short-side → 148
    transforms.CenterCrop(CROP_SIZE),              # deterministic 128×128 centre crop
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print('Transforms defined.')
print(f'  train: Resize({RESIZE_BEFORE_CROP}) → RandomCrop({CROP_SIZE}) → HFlip → VFlip → Rot90 → Normalize')
print(f'  val  : Resize({RESIZE_BEFORE_CROP}) → CenterCrop({CROP_SIZE}) → Normalize')

### 3.6 CNN Architecture *(exact replica from reference notebook)*

In [ ]:
class RegressionCNN(nn.Module):
    """
    Six conv blocks (BatchNorm + ReLU + MaxPool2d(2,2)), three FC layers, ELU output.
    Flattened size is computed automatically from img_height/img_width.
    """
    def __init__(self, input_channels=3, img_height=128, img_width=128, depth=16):
        super(RegressionCNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # Block 1  128→64
            nn.Conv2d(input_channels, depth, 3, padding=1),
            nn.BatchNorm2d(depth), nn.ReLU(),
            nn.Conv2d(depth, depth, 3, padding=1),
            nn.BatchNorm2d(depth), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 2  64→32
            nn.Conv2d(depth, depth*2, 3, padding=1),
            nn.BatchNorm2d(depth*2), nn.ReLU(),
            nn.Conv2d(depth*2, depth*2, 3, padding=1),
            nn.BatchNorm2d(depth*2), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 3  32→16
            nn.Conv2d(depth*2, depth*4, 3, padding=1),
            nn.BatchNorm2d(depth*4), nn.ReLU(),
            nn.Conv2d(depth*4, depth*4, 3, padding=1),
            nn.BatchNorm2d(depth*4), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 4  16→8
            nn.Conv2d(depth*4, depth*8, 3, padding=1),
            nn.BatchNorm2d(depth*8), nn.ReLU(),
            nn.Conv2d(depth*8, depth*8, 3, padding=1),
            nn.BatchNorm2d(depth*8), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 5  8→4
            nn.Conv2d(depth*8, depth*16, 3, padding=1),
            nn.BatchNorm2d(depth*16), nn.ReLU(),
            nn.Conv2d(depth*16, depth*16, 3, padding=1),
            nn.BatchNorm2d(depth*16), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 6  4→2
            nn.Conv2d(depth*16, depth*32, 3, padding=1),
            nn.BatchNorm2d(depth*32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, input_channels, img_height, img_width)
            self.flattened_size = self.conv_layers(dummy).view(1, -1).size(1)

        self.fc_layers = nn.Sequential(
            nn.Linear(self.flattened_size,      self.flattened_size // 2), nn.ReLU(),
            nn.Linear(self.flattened_size // 2, self.flattened_size // 4), nn.ReLU(),
            nn.Linear(self.flattened_size // 4, 1),
            nn.ELU()
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        return self.fc_layers(x)


_m = RegressionCNN(depth=16, img_height=CROP_SIZE, img_width=CROP_SIZE).to(device)
with torch.no_grad():
    _y = _m(torch.zeros(2, 3, CROP_SIZE, CROP_SIZE).to(device))
total_p = sum(p.numel() for p in _m.parameters())
print(f'Architecture OK  |  output: {tuple(_y.shape)}')
print(f'Flattened size  : {_m.flattened_size}')
print(f'Total params    : {total_p:,}')
del _m, _y

### 3.7 Metric Helper Functions

Primary metrics:
- **Accuracy@0** — fraction of predictions where `round(pred) == true`
- **Accuracy@1** — fraction where `|round(pred) − true| ≤ 1`
- **R²** — coefficient of determination on continuous predictions
- **RMSE** — root mean squared error on continuous predictions

In [ ]:
def compute_metrics(targets, preds):
    """
    Compute all four primary metrics.

    Parameters
    ----------
    targets : array-like  ground-truth scores (float, e.g. 1.0 … 8.0)
    preds   : array-like  continuous model predictions

    Returns
    -------
    dict with keys: acc0, acc1, r2, rmse
    """
    t = np.asarray(targets, dtype=float)
    p = np.asarray(preds,   dtype=float)

    p_rounded = np.round(p)

    acc0 = np.mean(p_rounded == t)
    acc1 = np.mean(np.abs(p_rounded - t) <= 1)
    r2   = r2_score(t, p) if len(np.unique(t)) > 1 else 0.0
    rmse = float(np.sqrt(mean_squared_error(t, p)))

    return {'acc0': acc0, 'acc1': acc1, 'r2': r2, 'rmse': rmse}


def metrics_str(m):
    return (f'Acc@0={m["acc0"]*100:.1f}%  '
            f'Acc@1={m["acc1"]*100:.1f}%  '
            f'R²={m["r2"]:.4f}  '
            f'RMSE={m["rmse"]:.4f}')


print('compute_metrics defined.')

### 3.8 Training and Validation Epoch Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, preds, tgts = 0.0, [], []

    for images, targets in tqdm(loader, desc='  train', leave=False):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        out  = model(images)
        loss = criterion(out, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds.extend(out.detach().cpu().numpy().flatten())
        tgts.extend(targets.detach().cpu().numpy().flatten())

    n  = len(loader.dataset)
    tr = compute_metrics(tgts, preds)
    return running_loss / n, tr


def validate_epoch(model, loader, criterion):
    model.eval()
    running_loss, preds, tgts = 0.0, [], []

    with torch.no_grad():
        for images, targets in tqdm(loader, desc='  val  ', leave=False):
            images, targets = images.to(device), targets.to(device)
            out  = model(images)
            loss = criterion(out, targets)
            running_loss += loss.item() * images.size(0)
            preds.extend(out.cpu().numpy().flatten())
            tgts.extend(targets.cpu().numpy().flatten())

    n  = len(loader.dataset)
    vl = compute_metrics(tgts, preds)
    return running_loss / n, vl, preds, tgts


print('train_epoch / validate_epoch defined.')

## 4. Visualise All Pixel Filters

One representative image processed through all 7 variants side-by-side.

In [ ]:
demo_class  = folder_numbers[len(folder_numbers) // 2]
demo_dir    = os.path.join(input_folder, FOLDER_NAME_TPL.format(n=demo_class))
cc_tok      = '+cc+' if use_cc else '+nocc+'
all_tifs    = [p for p in glob(os.path.join(demo_dir, '*.tif')) if os.path.isfile(p)]
has_tok     = any(cc_tok in p for p in all_tifs)
demo_paths  = [p for p in all_tifs if cc_tok in p] if has_tok else all_tifs

if demo_paths:
    demo_img = skio.imread(demo_paths[0])

    K_vals = [None, 0.5, 1.0, 1.5, 2.0, 3.0, 'zscore']
    labels = ['nocc\n(raw)', 'K=0.5', 'K=1.0', 'K=1.5', 'K=2.0', 'K=3.0',
              f'Z-score\n(|z|≤{ZSCORE_THRESHOLD})']

    fig, axes = plt.subplots(1, 7, figsize=(28, 5))
    for ax, K, lbl, col in zip(axes, K_vals, labels, DS_COLORS):
        if K == 'zscore':
            img_show = zscore_filter(demo_img, threshold=ZSCORE_THRESHOLD)
        elif K is not None:
            img_show = lab_statistical_filter(demo_img, K)
        else:
            img_show = demo_img.astype(np.float32) / 65535.0

        pct = (img_show.sum(axis=2) > 0).mean() * 100
        ax.imshow(np.clip(img_show, 0, 1))
        ax.set_title(f'{lbl}\n{pct:.1f}% pixels kept',
                     fontsize=10, fontweight='bold', color=col)
        ax.axis('off')

    plt.suptitle('Pixel Filter Comparison — same image, all 7 variants',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f'No demo images found in: {demo_dir}')

## 5. Train on Every Dataset

Same `train_list_master` / `val_list_master` split reused across all 7 variants.
Best model checkpoint selected by lowest validation **RMSE**.

In [ ]:
all_results = {}   # ds_label → dict

for ds_label, K in DATASETS:
    print('\n' + '='*80)
    print(f'  DATASET: {ds_label}   (K = {K})')
    print('='*80)
    print(f'  Train: {len(train_list_master)}  |  Val: {len(val_list_master)}')

    train_ds = ImageRegressionDataset(
        train_list_master, transform=train_transform, target_noise_std=0.15, K=K)
    val_ds = ImageRegressionDataset(
        val_list_master, transform=val_transform, target_noise_std=0.0, K=K)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)

    model = RegressionCNN(
        input_channels=3, img_height=CROP_SIZE, img_width=CROP_SIZE, depth=16
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0.0)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.1, patience=5, verbose=True)

    history = {k: [] for k in [
        'train_loss', 'val_loss',
        'train_acc0', 'val_acc0',
        'train_acc1', 'val_acc1',
        'train_r2',   'val_r2',
        'train_rmse', 'val_rmse',
        'lr'
    ]}

    safe_label      = ds_label.replace('=', '').replace('.', '')
    best_model_path = f'best_cnn_{safe_label}.pth'
    best_val_rmse   = float('inf')
    no_improve      = 0

    for epoch in range(num_epochs):
        tr_loss, tr_m             = train_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_m, vl_p, vl_t = validate_epoch(model, val_loader, criterion)

        scheduler.step(vl_loss)
        lr_now = optimizer.param_groups[0]['lr']

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        for key in ('acc0', 'acc1', 'r2', 'rmse'):
            history[f'train_{key}'].append(tr_m[key])
            history[f'val_{key}'].append(vl_m[key])
        history['lr'].append(lr_now)

        print(
            f'  [{epoch+1:4d}/{num_epochs}] '
            f'train RMSE {tr_m["rmse"]:.4f}  Acc@1 {tr_m["acc1"]*100:.1f}% | '
            f'val RMSE {vl_m["rmse"]:.4f}  Acc@1 {vl_m["acc1"]*100:.1f}%  '
            f'R² {vl_m["r2"]:.4f} | lr {lr_now:.2e}',
            end=''
        )

        if vl_m['rmse'] < best_val_rmse:
            best_val_rmse = vl_m['rmse']
            torch.save(model.state_dict(), best_model_path)
            no_improve = 0
            print('  ✓')
        else:
            no_improve += 1
            print(f'  ({no_improve}/{early_stopping_patience})')
            if no_improve >= early_stopping_patience:
                print(f'  ⚠ Early stopping at epoch {epoch+1}')
                break

    # ── Load best checkpoint and compute final metrics ──────────────────────
    model.load_state_dict(torch.load(best_model_path))
    _, final_m, final_preds, final_tgts = validate_epoch(model, val_loader, criterion)

    final_preds = np.array(final_preds).flatten()
    final_tgts  = np.array(final_tgts).flatten()

    all_results[ds_label] = {
        'history':     history,
        'val_preds':   final_preds,
        'val_targets': final_tgts,
        'acc0':        final_m['acc0'],
        'acc1':        final_m['acc1'],
        'r2':          final_m['r2'],
        'rmse':        final_m['rmse'],
        'n_epochs':    len(history['train_loss']),
        'model_path':  best_model_path,
        'K':           K,
    }

    print(f'  ── Final: {metrics_str(final_m)}')

print('\n✓ All datasets trained.')

## 6. Per-Dataset Learning Curves

Four panels per dataset: **RMSE**, **Accuracy@1**, **R²**, **Learning rate**.

In [ ]:
for ds_label, res in all_results.items():
    h  = res['history']
    ep = range(1, res['n_epochs'] + 1)

    fig, axes = plt.subplots(1, 4, figsize=(22, 4))
    fig.suptitle(
        f'Learning curves — {ds_label}   '
        f'[Acc@0={res["acc0"]*100:.1f}%  Acc@1={res["acc1"]*100:.1f}%  '
        f'R²={res["r2"]:.4f}  RMSE={res["rmse"]:.4f}]',
        fontsize=12, fontweight='bold')

    # ── RMSE ────────────────────────────────────────────────────────────────
    axes[0].plot(ep, h['train_rmse'], 'b-', lw=2, label='Train')
    axes[0].plot(ep, h['val_rmse'],   'r-', lw=2, label='Val')
    axes[0].set_title('RMSE'); axes[0].legend(); axes[0].grid(alpha=.3)
    axes[0].set_xlabel('Epoch')

    # ── Accuracy@1 ──────────────────────────────────────────────────────────
    axes[1].plot(ep, [v*100 for v in h['train_acc1']], 'b-', lw=2, label='Train')
    axes[1].plot(ep, [v*100 for v in h['val_acc1']],   'r-', lw=2, label='Val')
    axes[1].set_title('Accuracy@1 (%)'); axes[1].legend(); axes[1].grid(alpha=.3)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylim(0, 105)

    # ── R² ──────────────────────────────────────────────────────────────────
    axes[2].plot(ep, h['val_r2'], 'g-', lw=2, label='Val R²')
    axes[2].plot(ep, h['train_r2'], 'b--', lw=1.5, alpha=.6, label='Train R²')
    axes[2].set_title('R²'); axes[2].legend(); axes[2].grid(alpha=.3)
    axes[2].set_xlabel('Epoch')

    # ── Learning rate ────────────────────────────────────────────────────────
    axes[3].plot(ep, h['lr'], 'm-', lw=2)
    axes[3].set_title('Learning Rate'); axes[3].set_yscale('log')
    axes[3].grid(alpha=.3); axes[3].set_xlabel('Epoch')

    plt.tight_layout()
    plt.show()

## 7. Per-Dataset Predictions vs Ground Truth

Three panels per dataset: **scatter plot**, **residuals**, **per-class box plot**.

In [ ]:
for ds_label, res in all_results.items():
    vp, vt = res['val_preds'], res['val_targets']
    residuals = vp - vt

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    fig.suptitle(
        f'{ds_label}  —  '
        f'Acc@0={res["acc0"]*100:.1f}%  Acc@1={res["acc1"]*100:.1f}%  '
        f'R²={res["r2"]:.4f}  RMSE={res["rmse"]:.4f}',
        fontsize=12, fontweight='bold')

    # ── Scatter: predicted vs true ───────────────────────────────────────────
    ax = axes[0]
    ax.scatter(vt, vp, alpha=.65, s=45, edgecolors='k', lw=.4,
               c=np.abs(residuals), cmap='RdYlGn_r', vmin=0, vmax=2)
    lims = [min(vt.min(), vp.min()) - .5, max(vt.max(), vp.max()) + .5]
    ax.plot(lims, lims, 'r--', lw=2, label='Perfect')
    ax.fill_between(lims, [l - 1 for l in lims], [l + 1 for l in lims],
                    alpha=.12, color='green', label='±1 band')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('True score', fontweight='bold')
    ax.set_ylabel('Predicted score', fontweight='bold')
    ax.set_title('Scatter (colour = |error|)')
    ax.legend(); ax.grid(alpha=.3)

    # ── Residuals ────────────────────────────────────────────────────────────
    ax = axes[1]
    ax.scatter(vt, residuals, alpha=.65, s=45, edgecolors='k', lw=.4,
               c=np.abs(residuals), cmap='RdYlGn_r', vmin=0, vmax=2)
    ax.axhline(0,  color='r',     lw=2,   ls='--')
    ax.axhline(+1, color='green', lw=1.5, ls=':', alpha=.7, label='±1')
    ax.axhline(-1, color='green', lw=1.5, ls=':', alpha=.7)
    ax.set_xlabel('True score',    fontweight='bold')
    ax.set_ylabel('Pred − True',   fontweight='bold')
    ax.set_title('Residuals')
    ax.legend(); ax.grid(alpha=.3)

    # Annotate within-1 fraction
    w1 = (np.abs(residuals) <= 1).mean() * 100
    ax.text(0.02, 0.97, f'Within ±1: {w1:.1f}%',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', fc='wheat', alpha=.7))

    # ── Per-class box plot ────────────────────────────────────────────────────
    ax = axes[2]
    uniq = np.unique(vt)
    bp = ax.boxplot(
        [vp[vt == t] for t in uniq],
        positions=uniq, widths=.6,
        patch_artist=True,
        boxprops=dict(facecolor='lightblue', edgecolor='k', lw=1.5),
        medianprops=dict(color='red', lw=2),
        whiskerprops=dict(lw=1.2),
        capprops=dict(lw=1.2)
    )
    for t in uniq:
        ax.axhline(t, color='green', ls='--', alpha=.4, lw=1)
    ax.set_xlabel('True class', fontweight='bold')
    ax.set_ylabel('Predicted score', fontweight='bold')
    ax.set_title('Per-class distribution')
    ax.set_xticks(uniq)
    ax.grid(alpha=.3, axis='y')

    plt.tight_layout()
    plt.show()

## 8. Cross-Dataset Comparison

Summary table and bar charts for all four primary metrics across all 7 variants.

In [ ]:
rows = []
for ds_label, res in all_results.items():
    rows.append(dict(
        Dataset = ds_label,
        Epochs  = res['n_epochs'],
        Acc0_pct  = round(res['acc0'] * 100, 2),
        Acc1_pct  = round(res['acc1'] * 100, 2),
        R2        = round(res['r2'],   4),
        RMSE      = round(res['rmse'], 4),
    ))

summary_df = pd.DataFrame(rows).set_index('Dataset')
summary_df.index.name = 'Dataset'

print('\n── Validation summary ──────────────────────────────────────────────────────')
print(summary_df.to_string())
print()

# Highlight best in each metric column
best_row = pd.Series({
    'Acc0_pct': summary_df['Acc0_pct'].idxmax(),
    'Acc1_pct': summary_df['Acc1_pct'].idxmax(),
    'R2':       summary_df['R2'].idxmax(),
    'RMSE':     summary_df['RMSE'].idxmin(),    # lower is better
})
print('Best per metric:')
for metric, ds in best_row.items():
    val = summary_df.loc[ds, metric]
    print(f'  {metric:10s}: {ds:10s}  ({val})')

summary_df.to_csv('comparison_summary.csv')
print('\nSaved → comparison_summary.csv')

In [ ]:
datasets = list(summary_df.index)
x        = np.arange(len(datasets))
cols     = DS_COLORS[:len(datasets)]

metrics_cfg = [
    ('Acc0_pct', 'Accuracy@0 (%)',   False),   # (column, label, lower_is_better)
    ('Acc1_pct', 'Accuracy@1 (%)',   False),
    ('R2',       'R²',               False),
    ('RMSE',     'RMSE',             True),
]

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
fig.suptitle('Validation Metrics — All 7 Preprocessing Variants',
             fontsize=14, fontweight='bold')

for ax, (col, label, lower_better) in zip(axes, metrics_cfg):
    vals  = summary_df[col].values
    best_idx = vals.argmin() if lower_better else vals.argmax()

    bars = ax.bar(x, vals, color=cols, edgecolor='k', lw=0.8, zorder=3)

    # Gold star on the best bar
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(2.5)
    ax.text(x[best_idx], vals[best_idx] + max(vals) * 0.01,
            '★', ha='center', va='bottom', fontsize=13, color='goldenrod')

    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=25, ha='right', fontsize=9)
    ax.grid(alpha=.3, axis='y', zorder=0)

    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.005,
                f'{v:.2f}' if col not in ('Acc0_pct', 'Acc1_pct') else f'{v:.1f}',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 5))
fig.suptitle('Validation Curves — All Datasets', fontsize=14, fontweight='bold')

for (ds_label, res), col in zip(all_results.items(), DS_COLORS):
    ep = range(1, res['n_epochs'] + 1)
    h  = res['history']
    axes[0].plot(ep, h['val_rmse'],              lw=2, color=col, label=ds_label)
    axes[1].plot(ep, [v*100 for v in h['val_acc1']], lw=2, color=col, label=ds_label)
    axes[2].plot(ep, h['val_r2'],                lw=2, color=col, label=ds_label)

for ax, title, ylabel in zip(axes,
    ['Validation RMSE',        'Validation Accuracy@1 (%)', 'Validation R²'],
    ['RMSE (lower = better)',  'Acc@1 (%)',                 'R²']):
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=.3)

plt.tight_layout()
plt.show()

In [ ]:
unique_classes = np.unique(list(all_results.values())[0]['val_targets'])

# Build per-class Accuracy@1 matrix
acc1_matrix = {}
for ds_label, res in all_results.items():
    vp = np.round(res['val_preds'])
    vt = res['val_targets']
    acc1_matrix[ds_label] = [
        (np.abs(vp[vt == c] - c) <= 1).mean() * 100
        if (vt == c).any() else np.nan
        for c in unique_classes
    ]

acc1_df = pd.DataFrame(acc1_matrix,
                        index=[f'Score {int(c)}' for c in unique_classes])
print('Accuracy@1 (%) per class per dataset:')
print(acc1_df.round(1).to_string())

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(acc1_df.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
ax.set_xticks(range(len(acc1_df.columns)))
ax.set_xticklabels(acc1_df.columns, rotation=20, ha='right', fontsize=9)
ax.set_yticks(range(len(acc1_df.index)))
ax.set_yticklabels(acc1_df.index)

for i in range(acc1_df.shape[0]):
    for j in range(acc1_df.shape[1]):
        v = acc1_df.values[i, j]
        if not np.isnan(v):
            clr = 'white' if v < 40 else 'black'
            ax.text(j, i, f'{v:.0f}%', ha='center', va='center',
                    fontsize=9, fontweight='bold', color=clr)

plt.colorbar(im, ax=ax, label='Accuracy@1 (%)')
ax.set_title('Accuracy@1 per class × dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Save Results

In [ ]:
for ds_label, res in all_results.items():
    safe = ds_label.replace('=', '').replace('.', '')
    vp, vt = res['val_preds'], res['val_targets']
    vp_r   = np.round(vp)

    pd.DataFrame({
        'true':    vt,
        'pred':    vp,
        'pred_r':  vp_r,
        'abs_err': np.abs(vp - vt),
        'sq_err':  (vp - vt) ** 2,
        'exact':   (vp_r == vt).astype(int),
        'within1': (np.abs(vp_r - vt) <= 1).astype(int),
    }).to_csv(f'results_{safe}.csv', index=False)

    pd.DataFrame(res['history']).to_csv(f'history_{safe}.csv', index=False)
    print(f'  {ds_label}: results_{safe}.csv + history_{safe}.csv')

print('\n✓ All CSVs saved.')

## 10. Prediction Friezes

Friezes for **every** dataset variant (not just the best), showing the same JPG
images sorted by the model's **predicted** score and by **annotation**.

Each cell title shows the predicted score (rounded to 2 dp).  
Row label = true score class.  
The frieze title includes all four primary metrics for that variant.

In [ ]:
# ── JPG root ─────────────────────────────────────────────────────────────────
jpg_root = r'F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform'

print(f'Building JPG index under:\n  {jpg_root}')
jpg_index = {}
for dirpath, _, filenames in os.walk(jpg_root):
    if 'OUTPUT' in dirpath:
        continue
    for fn in filenames:
        if fn.lower().endswith(('.jpg', '.jpeg')):
            stem = os.path.splitext(fn)[0]
            jpg_index[stem] = os.path.join(dirpath, fn)

print(f'  {len(jpg_index)} JPG files indexed.')
if len(jpg_index) == 0:
    print('  ⚠  No JPGs found — check jpg_root path.')

# ── Collect images in val order (same for all variants) ──────────────────────
# We use the nocc model just to establish path order; images are loaded from JPG.
first_ds  = list(all_results.keys())[0]
first_K   = all_results[first_ds]['K']
_val_ds   = ImageRegressionDataset(val_list_master, transform=val_transform,
                                    target_noise_std=0.0, K=first_K)
_val_load  = DataLoader(_val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

path_order    = [p for p, _ in val_list_master]
all_true_arr  = np.array([t for _, t in val_list_master])

all_imgs_frieze = []
missing_jpg = 0
for tif_path in path_order:
    stem     = os.path.splitext(os.path.basename(tif_path))[0]
    jpg_path = jpg_index.get(stem)
    if jpg_path and os.path.isfile(jpg_path):
        img = np.array(Image.open(jpg_path).convert('RGB'), dtype=np.float32) / 255.0
    else:
        missing_jpg += 1
        raw = skio.imread(tif_path).astype(np.float32) / 65535.0
        img = raw
    all_imgs_frieze.append(img)

print(f'  {len(all_imgs_frieze)} images loaded  |  {missing_jpg} fallback to TIF')
if missing_jpg > 0:
    print('  ⚠  Some JPGs not found — check jpg_root or filename convention.')

score_labels_frieze = sorted(np.unique(all_true_arr))

In [ ]:
def draw_multi_row_frieze(images, pred_scores, true_scores, title,
                           score_labels, max_per_row=15, cell_size=1.8):
    """
    Multi-row frieze.  Rows are defined by `score_labels` (true classes).
    Within each row the images are ordered by `pred_scores`.
    Each cell title shows the predicted score; colour of the title indicates
    accuracy:  green = exact match, orange = within-1, red = off by >1.
    """
    images      = list(images)
    pred_scores = np.asarray(pred_scores, dtype=float)
    true_scores = np.asarray(true_scores, dtype=float)
    pred_rounded = np.round(pred_scores)

    score_rows = {}
    for sl in score_labels:
        mask = true_scores == float(sl)
        if not mask.any():
            score_rows[sl] = []
            continue
        idx = np.where(mask)[0]
        score_rows[sl] = list(idx[np.argsort(pred_scores[idx])])

    display_rows = []
    for sl in score_labels:
        indices = score_rows[sl]
        if not indices:
            display_rows.append((sl, []))
            continue
        for chunk_start in range(0, len(indices), max_per_row):
            display_rows.append((sl, indices[chunk_start:chunk_start + max_per_row]))

    n_rows = len(display_rows)
    n_cols = max_per_row + 1

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * cell_size, n_rows * cell_size),
        squeeze=False
    )
    fig.suptitle(title, fontsize=12, fontweight='bold', y=1.01)

    for row_idx, (sl, indices) in enumerate(display_rows):
        ax_lbl = axes[row_idx, 0]
        ax_lbl.axis('off')
        ax_lbl.text(0.5, 0.5, str(int(sl)),
                    fontsize=20, fontweight='bold',
                    ha='center', va='center',
                    transform=ax_lbl.transAxes, color='#2c3e50')

        for col_off, img_idx in enumerate(indices):
            ax  = axes[row_idx, col_off + 1]
            ax.imshow(np.clip(images[img_idx], 0, 1))

            pred_val = pred_scores[img_idx]
            err      = abs(pred_rounded[img_idx] - true_scores[img_idx])
            title_col = ('#27ae60' if err == 0      # exact match → green
                         else '#e67e22' if err <= 1  # within-1   → orange
                         else '#e74c3c')             # >1 error   → red

            ax.set_title(f'{pred_val:.2f}', fontsize=7, pad=2, color=title_col,
                         fontweight='bold')
            ax.axis('off')

        for col_off in range(len(indices), max_per_row):
            axes[row_idx, col_off + 1].axis('off')

    plt.tight_layout()
    plt.show()


print('draw_multi_row_frieze defined.')

In [ ]:
for ds_label, K in DATASETS:
    print(f'\n{"="*70}')
    print(f'  FRIEZE: {ds_label}')
    print(f'{"="*70}')

    safe_label = ds_label.replace('=', '').replace('.', '')
    m = RegressionCNN(
        input_channels=3, img_height=CROP_SIZE, img_width=CROP_SIZE, depth=16
    ).to(device)
    m.load_state_dict(torch.load(all_results[ds_label]['model_path']))
    m.eval()

    val_ds_v  = ImageRegressionDataset(
        val_list_master, transform=val_transform, target_noise_std=0.0, K=K)
    val_load_v = DataLoader(val_ds_v, batch_size=batch_size, shuffle=False, num_workers=0)

    preds_v = []
    with torch.no_grad():
        for imgs, _ in tqdm(val_load_v, desc=f'  inferring {ds_label}', leave=False):
            out = m(imgs.to(device)).cpu()
            preds_v.extend(out.numpy())
    preds_v = np.array(preds_v)[:, 0]

    r = all_results[ds_label]
    hdr = (f'[{ds_label}]  '
           f'Acc@0={r["acc0"]*100:.1f}%  Acc@1={r["acc1"]*100:.1f}%  '
           f'R²={r["r2"]:.4f}  RMSE={r["rmse"]:.4f}')

    # ── Sorted by PREDICTION (rows = true class, within-row sorted by pred) ──
    draw_multi_row_frieze(
        images      = all_imgs_frieze,
        pred_scores = preds_v,
        true_scores = all_true_arr,
        title       = f'{hdr}\n→ Sorted by PREDICTION',
        score_labels= score_labels_frieze,
        max_per_row = 15,
        cell_size   = 1.8
    )

    # ── Sorted by ANNOTATION (rows = true class, within-row sorted by true) ──
    draw_multi_row_frieze(
        images      = all_imgs_frieze,
        pred_scores = preds_v,
        true_scores = all_true_arr,
        title       = f'{hdr}\n→ Sorted by ANNOTATION',
        score_labels= score_labels_frieze,
        max_per_row = 15,
        cell_size   = 1.8
    )

    del m